
# FL - DATASET: HAR (v01)

### J. Reyes-Ortiz, D. Anguita, A. Ghio, L. Oneto, and X. Parra. "Human Activity Recognition Using Smartphones," UCI Machine Learning Repository, 2013. [Online]. Available: https://doi.org/10.24432/C54S4K.



In [15]:
# ID = 00 - Download e extração do dataset UCI-HAR
import os
import zipfile
import urllib.request

DATASET_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
DATASET_ZIP = "/content/UCI_HAR.zip"
DATASET_ROOT = "/content/UCI HAR Dataset"

print("Baixando o pacote compactado UCI HAR...")
urllib.request.urlretrieve(DATASET_URL, DATASET_ZIP)

print("Extraindo o arquivo...")
with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
    zip_ref.extractall("/content")

inner_zip = "/content/UCI HAR Dataset.zip"
if os.path.exists(inner_zip):
    with zipfile.ZipFile(inner_zip, "r") as zip_ref:
        zip_ref.extractall("/content")

print("✅ Dataset pronto em:", DATASET_ROOT)

Baixando o pacote compactado UCI HAR...
Extraindo o arquivo principal...
Extraindo o zip interno do dataset...
✅ Dataset pronto em: /content/UCI HAR Dataset


In [16]:
#ID = 01 — Imports globais
# -------------------------------------------------
import os
import json
import random
import time
import copy
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from torchvision import datasets, transforms
from tqdm.auto import tqdm

from datetime import datetime

# Fuzzy
!pip install scikit-fuzzy
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [17]:
# ID = 02 — Seed, device, configuração global e diretórios (FL-HAR only)
# ---------------------------------------------------------------------

# Reprodutibilidade
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SEED={SEED}, DEVICE={DEVICE}")

# -----------------------------
# Metadados do experimento
# -----------------------------
EXPERIMENT_CFG = {
    "experiment_id": "fl_har_9ch_baseline",
    "dataset": "UCI_HAR",
    "dataset_root": "/content/UCI HAR Dataset",
    "signal_set": "all_9ch",
    "num_workers": 0,
}

EXPERIMENT_ID = EXPERIMENT_CFG["experiment_id"]

# -----------------------------
# Hiperparâmetros do FL
# -----------------------------
FL_CFG = {       ## Ajuste para o experimento
    "n_rounds": 10,
    "local_epochs": 5,
    "lr": 1e-4,
    "weight_decay": 1e-2,
    "batch_size_train": 32,
    "batch_size_eval": 64,
    "max_grad_norm": 5.0,
    "fedprox_mu": 0.01,
}

N_ROUNDS = FL_CFG["n_rounds"]
LOCAL_EPOCHS = FL_CFG["local_epochs"]
LR = FL_CFG["lr"]
WEIGHT_DECAY = FL_CFG["weight_decay"]
BATCH_SIZE_TRAIN = FL_CFG["batch_size_train"]
BATCH_SIZE_EVAL = FL_CFG["batch_size_eval"]
MAX_GRAD_NORM = FL_CFG["max_grad_norm"]
FEDPROX_MU = FL_CFG["fedprox_mu"]



# -----------------------------
# Diretórios de resultados
# -----------------------------
from datetime import datetime
from pathlib import Path

BASE_RESULTS_DIR = Path("/content/FL_Fuzzy_results")
METRICS_ROOT = BASE_RESULTS_DIR / "metricas"
RUN_ID = datetime.now().strftime("Exp_%Y%m%d_%H%M%S")
EXPERIMENT_DIR = METRICS_ROOT / RUN_ID
GRAPHICS_DIR = EXPERIMENT_DIR / "graficos"
SPREADSHEETS_DIR = EXPERIMENT_DIR / "arquivos_csv"

for d in [
    METRICS_ROOT,
    EXPERIMENT_DIR,
    GRAPHICS_DIR,
    SPREADSHEETS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("Experimento:", RUN_ID)
print("Graficos:", GRAPHICS_DIR)
print("Planilhas:", SPREADSHEETS_DIR)


# -----------------------------
# Resumo da configuração
# -----------------------------
print("\nResumo da configuração ativa:")
print(f"- dataset          : {EXPERIMENT_CFG['dataset']}")
print(f"- signal_set       : {EXPERIMENT_CFG['signal_set']}")
print(f"- n_rounds         : {N_ROUNDS}")
print(f"- local_epochs     : {LOCAL_EPOCHS}")
print(f"- lr               : {LR}")
print(f"- weight_decay     : {WEIGHT_DECAY}")
print(f"- batch_size_train : {BATCH_SIZE_TRAIN}")
print(f"- batch_size_eval  : {BATCH_SIZE_EVAL}")
print(f"- fedprox_mu       : {FEDPROX_MU}")

SEED=42, DEVICE=cuda
Experimento: Exp_20260926_184858
Graficos: /content/FL_Fuzzy_results/metricas/Exp_20260926_184858/graficos
Planilhas: /content/FL_Fuzzy_results/metricas/Exp_20260926_184858/arquivos_csv

Resumo da configuração ativa:
- dataset          : UCI_HAR
- signal_set       : all_9ch
- n_rounds         : 10
- local_epochs     : 5
- lr               : 0.0001
- weight_decay     : 0.01
- batch_size_train : 32
- batch_size_eval  : 64
- fedprox_mu       : 0.01


In [18]:
#ID = 03 — Adaptação da taxa de aprendizado com Fuzzy
# -------------------------------------------------

def learning_rate_fuzzy(f1,loss):

  x_f1 = np.arange(0, 1.01, 0.01)
  x_loss = np.arange(0, 1.01, 0.01)
  x_learning_rate = np.linspace(1e-4, 1e-2, 100)

  #variaveis de entrada
  f1_lr= ctrl.Antecedent(x_f1, 'f1')
  loss_lr = ctrl.Antecedent(x_loss, 'loss')

  #variaveis de saida
  learning_rate = ctrl.Consequent(x_learning_rate, 'learning_rate')

  f1_lr['small'] = fuzz.zmf(x_f1, 0, 0.3)
  f1_lr['medium'] = fuzz.trimf(x_f1, [0.2, 0.5, 0.8])
  f1_lr['high'] = fuzz.smf(x_f1, 0.7, 1)

  loss_lr['good'] = fuzz.zmf(x_loss, 0.06, 0.5)
  loss_lr['fair'] = fuzz.trimf(x_loss,[0.3, 0.5, 0.8])
  loss_lr['poor'] = fuzz.smf(x_loss,0.6,1.0)

  learning_rate['low'] = fuzz.zmf(x_learning_rate,1e-4,3.5e-3)
  learning_rate['medium'] = fuzz.trimf(x_learning_rate,[2e-3, 5e-3, 8e-3])
  learning_rate['high'] = fuzz.smf(x_learning_rate,6.5e-3,1e-2)

  # Visualização das funções de pertinencia
  # f1_lr.view()
  # loss_lr.view()
  # learning_rate.view()

  # Regras
  rule1  = ctrl.Rule(f1_lr['small'] & loss_lr['good'],       learning_rate['medium'])
  rule2  = ctrl.Rule(f1_lr['small'] & loss_lr['fair'],  learning_rate['high'])
  rule3  = ctrl.Rule(f1_lr['small'] & loss_lr['poor'],      learning_rate['high'])

  rule4  = ctrl.Rule(f1_lr['medium'] & loss_lr['good'],       learning_rate['low'])
  rule5  = ctrl.Rule(f1_lr['medium'] & loss_lr['fair'],  learning_rate['medium'])
  rule6  = ctrl.Rule(f1_lr['medium'] & loss_lr['poor'],      learning_rate['high'])

  rule7 = ctrl.Rule(f1_lr['high'] & loss_lr['good'],       learning_rate['low'])
  rule8 = ctrl.Rule(f1_lr['high'] & loss_lr['fair'],  learning_rate['medium'])
  rule9 = ctrl.Rule(f1_lr['high'] & loss_lr['poor'],      learning_rate['low'])

  # Montagem do sistema
  SE_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
  SE = ctrl.ControlSystemSimulation(SE_ctrl)

  # Entradas
  SE.input['f1'] = f1
  SE.input['loss'] = loss

  # Taxa de aprendizado gerada
  SE.compute()
  #SE.print_state()
  resultado = SE.output['learning_rate']

  # Visualização do resultado
  # learning_rate.view(sim=SE)
  return resultado

In [19]:
#ID = 04 — Adaptação das épocas locais com Fuzzy
# -------------------------------------------------

def epochs_fuzzy(gap,loss):
#vetores com os limites das variaveis de entrada
  x_gap = np.arange(-1, 1.01, 0.01)
  x_loss = np.arange(0, 1.1, 0.01)

  x_epoch = np.arange(4, 17, 1)

  #variaveis de entrada / aparece no if da regra
  gap_ep= ctrl.Antecedent(x_gap, 'gap')
  loss_ep = ctrl.Antecedent(x_loss, 'loss')

  #variaveis de saida
  epoch = ctrl.Consequent(x_epoch, 'epoch') # aparece no then


  gap_ep['negative'] = fuzz.zmf(x_gap, -0.4, 0.0)
  gap_ep['near_zero'] = fuzz.trimf(x_gap,[-0.2, 0.0, 0.2])
  gap_ep['positive'] = fuzz.smf(x_gap, 0.0, 0.4)

  loss_ep['good'] = fuzz.zmf(x_loss, 0.06, 0.5)
  loss_ep['fair'] = fuzz.trimf(x_loss,[0.3, 0.5, 0.8])
  loss_ep['poor'] = fuzz.smf(x_loss,0.6,1.0)

  epoch['low'] = fuzz.zmf(x_epoch, 4, 9)
  epoch['medium'] = fuzz.trimf(x_epoch,[6, 10, 14])
  epoch['high'] = fuzz.smf(x_epoch, 11, 16)

  # Visualização das funções de pertinencia
  # loss_ep.view()
  # gap_ep.view()
  # epoch.view()

  rule1  = ctrl.Rule(gap_ep['negative'] & loss_ep['good'],       epoch['medium'])
  rule2 = ctrl.Rule(gap_ep['negative'] & (loss_ep['fair'] | loss_ep['poor']), epoch['high'])
  rule3  = ctrl.Rule(gap_ep['near_zero'] & loss_ep['good'],       epoch['low'])
  rule4  = ctrl.Rule(gap_ep['near_zero'] & loss_ep['fair'],  epoch['medium'])
  rule5  = ctrl.Rule(gap_ep['near_zero'] & loss_ep['poor'],      epoch['high'])
  rule6 = ctrl.Rule(gap_ep['positive'] & (loss_ep['good'] | loss_ep['fair']), epoch['low'])
  rule7 = ctrl.Rule(gap_ep['positive'] & loss_ep['poor'],      epoch['medium'])

  # Montagem do sistema
  SE_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7])
  SE = ctrl.ControlSystemSimulation(SE_ctrl)

  # Entradas
  SE.input['gap'] = gap
  SE.input['loss'] = loss

  # Taxa de aprendizado gerada
  SE.compute()
  #SE.print_state()
  resultado = SE.output['epoch']

  # Visualização do resultado
  # epoch.view(sim=SE)
  return resultado


In [20]:
# ID = 05 — Função de avaliação de modelo
# ---------------------------------------

def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    device: str = DEVICE,
    return_report: bool = False,
    return_confusion: bool = False,
    average: str = "macro",         # mantido para compatibilidade; calculo de macro e weighted
    topk: tuple = (),               # ex.: (1, 5) para acc@1 e acc@5; vazio = não calcula
    labels: list | None = None,     # lista de rótulos/ordem (0..C-1); None tenta inferir
) -> dict:
    """
    Avalia um modelo num dataloader e retorna um dicionário com:

      - 'acc'
      - 'f1_macro', 'f1_weighted'
      - 'precision_macro', 'recall_macro'
      - 'precision_weighted', 'recall_weighted'

    Opcionalmente:
      - 'report' (classification_report como string) se return_report=True
      - 'confusion_matrix' (np.ndarray CxC) se return_confusion=True
      - 'topk': {'acc@k': float, ...} se topk não for vazio
    """
    was_training = model.training
    model.to(device)
    model.eval()

    all_preds, all_targets = [], []

    with torch.inference_mode():
        for xb, yb in dataloader:
            xb = xb.to(device, non_blocking=False)
            yb = yb.to(device, non_blocking=False)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            all_preds.append(preds.detach().cpu())
            all_targets.append(yb.detach().cpu())

    if not all_preds:
        raise RuntimeError("evaluate_model recebeu um dataloader vazio (sem batches).")

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_targets).numpy()

    # Métricas básicas
    acc  = accuracy_score(y_true, y_pred)
    f1m  = f1_score(y_true, y_pred, average="macro")
    f1w  = f1_score(y_true, y_pred, average="weighted")

    # Precision/Recall em macro & weighted
    p_m, r_m, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_w, r_w, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    out = {
        "acc": float(acc),
        "f1_macro": float(f1m),
        "f1_weighted": float(f1w),
        "precision_macro": float(p_m),
        "precision_weighted": float(p_w),
        "recall_macro": float(r_m),
        "recall_weighted": float(r_w),
    }

    # Top-K accuracy opcional
    if topk:
        topk = tuple(sorted({int(k) for k in topk if int(k) >= 1}))
        if topk:
            tk_hits = {f"acc@{k}": 0 for k in topk}
            total = 0
            with torch.inference_mode():
                for xb, yb in dataloader:
                    xb = xb.to(device, non_blocking=False)
                    yb = yb.to(device, non_blocking=False)
                    scores = model(xb)
                    total += yb.size(0)
                    max_k = max(topk)
                    _, topk_idx = scores.topk(k=max_k, dim=1, largest=True, sorted=True)
                    for k in topk:
                        hits = (topk_idx[:, :k] == yb.view(-1, 1)).any(dim=1).sum().item()
                        tk_hits[f"acc@{k}"] += hits
            out["topk"] = {k: tk_hits[f"acc@{k}"] / total for k in tk_hits}

    if return_report:
        out["report"] = classification_report(
            y_true, y_pred, digits=4, zero_division=0
        )

    if return_confusion:
        # se labels não vierem, infere ordenando os presentes
        if labels is None:
            labels = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        out["confusion_matrix"] = cm

    if was_training:
        model.train()

    return out


In [21]:
# ID = 06 — Índice de Jain
# -------------------------

def jain_index(x):
    """
    Calcula o índice de Jain para um vetor de valores não negativos.

    Fórmula:
        J(x) = (sum(x)^2) / (n * sum(x^2))

    Retorna valor em [0, 1], onde:
    - 1   -> distribuição perfeitamente uniforme
    - baixo -> maior desigualdade entre os elementos
    """
    x = np.asarray(x, dtype=np.float64)

    if x.size == 0:
        return float("nan")

    if np.any(x < 0):
        raise ValueError("jain_index requer valores não negativos.")

    denom = x.size * np.sum(x ** 2)

    if denom == 0:
        return float("nan")

    return float((np.sum(x) ** 2) / denom)

In [22]:
# ID = 07 — Dataset HAR
# -------------------------------------------------
class HARDataset(torch.utils.data.Dataset):
    """
    Dataset genérico para Human Activity Recognition (HAR).

    Parâmetros
    ----------
    X : np.ndarray ou torch.Tensor
        Dados de entrada no formato:
        - (N, T, S)  se input_layout="time_first"
        - (N, S, T)  se input_layout="channels_first"
    y : np.ndarray ou torch.Tensor
        Rótulos das atividades (N,)
    user_ids : np.ndarray ou torch.Tensor
        Identificador do usuário para cada janela (N,)
    tensor_format : str
        Formato retornado por __getitem__:
        - "channels_first": (S, T)
        - "flat":           (S*T,)
        - "cnn2d":          (1, S, T)
    input_layout : str
        Layout em que X já chega:
        - "time_first"      -> (N, T, S)
        - "channels_first"  -> (N, S, T)
    dtype : torch.dtype
        Tipo do tensor retornado
    """

    def __init__(
        self,
        X,
        y,
        user_ids,
        tensor_format: str = "channels_first",
        input_layout: str = "time_first",
        dtype: torch.dtype = torch.float32,
    ):
        super().__init__()

        assert len(X) == len(y) == len(user_ids), \
            "X, y e user_ids devem ter o mesmo comprimento"

        if isinstance(X, np.ndarray):
            X = torch.from_numpy(X)
        if isinstance(y, np.ndarray):
            y = torch.from_numpy(y)
        if isinstance(user_ids, np.ndarray):
            user_ids = torch.from_numpy(user_ids)

        self.X = X.to(dtype=dtype)
        self.y = y.long()
        self.user_ids = user_ids.long()
        self.tensor_format = tensor_format
        self.input_layout = input_layout

        if self.X.ndim != 3:
            raise ValueError(f"X deve ser 3D, mas recebeu shape={tuple(self.X.shape)}")

        self.N = self.X.shape[0]

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        Xw = self.X[idx]
        y = self.y[idx]

        # Normaliza primeiro para (S, T)
        if self.input_layout == "time_first":
            # Xw vem como (T, S)
            X_cf = Xw.transpose(0, 1)   # -> (S, T)
        elif self.input_layout == "channels_first":
            # Xw já vem como (S, T)
            X_cf = Xw
        else:
            raise ValueError(f"input_layout desconhecido: {self.input_layout}")

        if self.tensor_format == "channels_first":
            out = X_cf                       # (S, T)

        elif self.tensor_format == "flat":
            out = X_cf.reshape(-1)           # (S*T,)

        elif self.tensor_format == "cnn2d":
            out = X_cf.unsqueeze(0)          # (1, S, T)

        else:
            raise ValueError(
                f"tensor_format desconhecido: {self.tensor_format}"
            )

        return out, y

In [23]:
# ID = 08 — Carregamento dos sinais inerciais do UCI-HAR
# -----------------------------------------------------

def load_inertial_signals_uci_har(base_path, split, signal_set="all_9ch"):
    """
    Carrega os sinais inerciais do UCI-HAR.

    Parâmetros
    ----------
    base_path : str
        Caminho raiz do dataset UCI HAR Dataset.
    split : str
        "train" ou "test".
    signal_set : str
        "body_acc_3ch" -> usa apenas body_acc_{x,y,z}
        "all_9ch"      -> usa body_acc, body_gyro e total_acc

    Retorno
    -------
    X : np.ndarray
        Shape = (N, T, C)
        N = número de amostras
        T = número de timesteps (128)
        C = número de canais
    """
    inertial_dir = os.path.join(base_path, split, "Inertial Signals")

    if signal_set == "body_acc_3ch":
        signal_files = [
            f"body_acc_x_{split}.txt",
            f"body_acc_y_{split}.txt",
            f"body_acc_z_{split}.txt",
        ]
    elif signal_set == "all_9ch":
        signal_files = [
            f"body_acc_x_{split}.txt",
            f"body_acc_y_{split}.txt",
            f"body_acc_z_{split}.txt",
            f"body_gyro_x_{split}.txt",
            f"body_gyro_y_{split}.txt",
            f"body_gyro_z_{split}.txt",
            f"total_acc_x_{split}.txt",
            f"total_acc_y_{split}.txt",
            f"total_acc_z_{split}.txt",
        ]
    else:
        raise ValueError(
            f"signal_set inválido: {signal_set}. "
            f"Use 'body_acc_3ch' ou 'all_9ch'."
        )

    signals = []
    for fname in signal_files:
        fpath = os.path.join(inertial_dir, fname)

        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Arquivo não encontrado: {fpath}")

        data = np.loadtxt(fpath)   # shape: (N, 128)
        signals.append(data)

    # Empilha no último eixo -> (N, 128, C)
    X = np.stack(signals, axis=-1).astype(np.float32)

    print(
        f"[UCI-HAR] split={split} | signal_set={signal_set} "
        f"| X.shape={X.shape}"
    )

    return X

In [24]:
# ID = 09 — Loader consolidado do UCI-HAR
# ---------------------------------------

def load_uci_har_dataset(
    dataset_root,
    tensor_format="channels_first",
    signal_set="all_9ch",
):
    """
    Carrega o dataset UCI-HAR e devolve um DatasetBundle padronizado.
    """

    # -----------------------------
    # Carregamento dos sinais
    # -----------------------------
    X_train = load_inertial_signals_uci_har(
        dataset_root,
        split="train",
        signal_set=signal_set,
    )  # (N, T, C)

    X_test = load_inertial_signals_uci_har(
        dataset_root,
        split="test",
        signal_set=signal_set,
    )  # (N, T, C)

    # -----------------------------
    # Labels e IDs de usuário
    # -----------------------------
    y_train = np.loadtxt(
        os.path.join(dataset_root, "train", "y_train.txt"),
        dtype=np.int64
    ) - 1

    y_test = np.loadtxt(
        os.path.join(dataset_root, "test", "y_test.txt"),
        dtype=np.int64
    ) - 1

    subject_train = np.loadtxt(
        os.path.join(dataset_root, "train", "subject_train.txt"),
        dtype=np.int64
    )

    subject_test = np.loadtxt(
        os.path.join(dataset_root, "test", "subject_test.txt"),
        dtype=np.int64
    )

    # -----------------------------
    # Formato do tensor
    # -----------------------------
    if tensor_format == "channels_first":
        X_train = np.transpose(X_train, (0, 2, 1))  # (N, C, T)
        X_test = np.transpose(X_test, (0, 2, 1))    # (N, C, T)
        input_layout = "channels_first"
    elif tensor_format == "channels_last":
        input_layout = "time_first"
    else:
        raise ValueError(
            f"tensor_format inválido: {tensor_format}. "
            f"Use 'channels_first' ou 'channels_last'."
        )

    # -----------------------------
    # Datasets PyTorch
    # -----------------------------
    train_dataset = HARDataset(
        X_train,
        y_train,
        user_ids=subject_train,
        tensor_format="channels_first",
        input_layout=input_layout,
    )

    test_dataset = HARDataset(
        X_test,
        y_test,
        user_ids=subject_test,
        tensor_format="channels_first",
        input_layout=input_layout,
    )

    # -----------------------------
    # Dataset completo
    # -----------------------------
    X_full = np.concatenate([X_train, X_test], axis=0)
    y_full = np.concatenate([y_train, y_test], axis=0)
    subject_full = np.concatenate([subject_train, subject_test], axis=0)

    full_dataset = HARDataset(
        X_full,
        y_full,
        user_ids=subject_full,
        tensor_format="channels_first",
        input_layout=input_layout,
    )

    # -----------------------------
    # Metadados
    # -----------------------------
    n_classes = int(len(np.unique(y_full)))
    input_shape = tuple(X_train.shape[1:])

    metadata = {
        "dataset_name": "UCI_HAR",
        "uses_official_split": True,
        "signal_set": signal_set,
        "tensor_format": tensor_format,
        "n_input_channels": int(X_train.shape[1] if tensor_format == "channels_first" else X_train.shape[2]),
        "window_length": int(X_train.shape[2] if tensor_format == "channels_first" else X_train.shape[1]),
        "n_train_samples": int(len(train_dataset)),
        "n_test_samples": int(len(test_dataset)),
        "n_total_samples": int(len(full_dataset)),
        "n_subjects_train": int(len(np.unique(subject_train))),
        "n_subjects_test": int(len(np.unique(subject_test))),
    }

    print("✅ UCI-HAR carregado com sucesso")
    print(f" - signal_set     : {signal_set}")
    print(f" - tensor_format  : {tensor_format}")
    print(f" - X_train.shape  : {X_train.shape}")
    print(f" - X_test.shape   : {X_test.shape}")
    print(f" - input_shape    : {input_shape}")
    print(f" - n_classes      : {n_classes}")

    train_indices = np.arange(len(train_dataset), dtype=int)
    test_indices = np.arange(len(test_dataset), dtype=int)

    return DatasetBundle(
        full_dataset=full_dataset,
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        train_indices=train_indices,
        test_indices=test_indices,
        n_classes=n_classes,
        input_shape=input_shape,
        metadata=metadata,
    )

In [25]:
# ID = 10 — Carregamento do dataset ativo (FL-HAR only)
# -----------------------------------------------------

def load_active_dataset(cfg):
    """
    Carrega o dataset ativo do notebook.
    Nesta versão, o notebook suporta apenas UCI-HAR.
    """

    dataset_name = cfg["dataset"]

    if dataset_name != "UCI_HAR":
        raise ValueError(
            f"Dataset não suportado neste notebook: {dataset_name}. "
            f"Use apenas 'UCI_HAR'."
        )

    # -------------------------------------------------
    # Carrega o UCI-HAR consolidado
    # -------------------------------------------------
    bundle = load_uci_har_dataset(
        dataset_root=cfg["dataset_root"],
        tensor_format="channels_first",
        signal_set=cfg["signal_set"],
    )

    full_dataset = bundle.full_dataset
    train_dataset_global = bundle.train_dataset
    test_dataset_global = bundle.test_dataset
    n_classes = bundle.n_classes
    input_shape = bundle.input_shape
    metadata = dict(bundle.metadata)

    # -------------------------------------------------
    # Criação dos clientes federados a partir do TRAIN
    # -------------------------------------------------
    client_subsets_train = []
    client_subsets_test = []
    client_ids = []

    train_user_ids = np.asarray(train_dataset_global.user_ids)
    train_labels = np.asarray(train_dataset_global.y)

    unique_users = sorted(np.unique(train_user_ids).tolist())

    train_indices_global = []
    test_indices_global = []

    for uid in unique_users:
        user_idx = np.where(train_user_ids == uid)[0]

        # ignora clientes sem amostras suficientes
        if len(user_idx) < 2:
            print(f"⚠️ Usuário {uid} ignorado: poucas amostras ({len(user_idx)})")
            continue

        y_user = train_labels[user_idx]

        # tenta split estratificado por classe dentro do cliente
        try:
            idx_tr, idx_te = train_test_split(
                user_idx,
                test_size=0.2,
                random_state=SEED,
                stratify=y_user,
            )
        except ValueError:
            idx_tr, idx_te = train_test_split(
                user_idx,
                test_size=0.2,
                random_state=SEED,
                shuffle=True,
            )

        if len(idx_tr) == 0 or len(idx_te) == 0:
            print(f"⚠️ Usuário {uid} ignorado após split vazio.")
            continue

        client_ids.append(int(uid))
        client_subsets_train.append(Subset(train_dataset_global, idx_tr.tolist()))
        client_subsets_test.append(Subset(train_dataset_global, idx_te.tolist()))

        train_indices_global.extend(idx_tr.tolist())
        test_indices_global.extend(idx_te.tolist())

    train_indices_global = np.array(train_indices_global, dtype=int)
    test_indices_global = np.array(test_indices_global, dtype=int)

    # -------------------------------------------------
    # Avaliação global opcional
    # -------------------------------------------------
    optional_global_eval_loader_dataset = test_dataset_global

    metadata.update({
        "client_protocol": "local_split_within_official_train",
        "global_eval_protocol": "official_test_as_optional_global_eval",
        "uses_official_test_for_client_eval": False,
        "uses_official_test_for_global_eval": True,
        "n_clients": len(client_ids),
    })

    print("✅ Dataset ativo carregado com sucesso")
    print(f" - dataset_name           : {dataset_name}")
    print(f" - input_shape            : {input_shape}")
    print(f" - n_classes              : {n_classes}")
    print(f" - signal_set             : {cfg['signal_set']}")
    print(f" - n_clients              : {len(client_ids)}")
    print(f" - train_global_samples   : {len(train_dataset_global)}")
    print(f" - test_global_samples    : {len(test_dataset_global)}")

    return {
        "dataset_name": dataset_name,
        "full_dataset": full_dataset,
        "train_dataset_global": train_dataset_global,
        "test_dataset_global": test_dataset_global,
        "client_subsets_train": client_subsets_train,
        "client_subsets_test": client_subsets_test,
        "client_ids": client_ids,
        "n_classes": n_classes,
        "input_shape": input_shape,
        "metadata": metadata,
        "train_indices_global": train_indices_global,
        "test_indices_global": test_indices_global,
        "optional_global_eval_loader_dataset": optional_global_eval_loader_dataset,
    }

In [26]:

# ID = 11 — loaders consolidados (UCI-HAR)
# -------------------------------------------------------------

from dataclasses import dataclass
from collections import Counter
import os
import re
import numpy as np
import pandas as pd
from torch.utils.data import Subset


@dataclass
class DatasetBundle:
    train_dataset: object
    test_dataset: object
    full_dataset: object
    train_indices: np.ndarray
    test_indices: np.ndarray
    n_classes: int
    input_shape: tuple
    metadata: dict


In [27]:
# ID = 12 — Unpack da interface retornada por load_active_dataset
# ----------------------------------------------------------------

ds = load_active_dataset(EXPERIMENT_CFG)

full_dataset = ds["full_dataset"]
train_dataset_global = ds["train_dataset_global"]
test_dataset_global = ds["test_dataset_global"]

client_subsets_train = ds["client_subsets_train"]
client_subsets_test = ds["client_subsets_test"]
client_ids = ds["client_ids"]

N_CLASSES = ds["n_classes"]
INPUT_SHAPE = ds["input_shape"]
DATASET_METADATA = ds["metadata"]

train_indices_global = ds["train_indices_global"]
test_indices_global = ds["test_indices_global"]

optional_global_eval_loader_dataset = ds["optional_global_eval_loader_dataset"]

print("✅ Interface do dataset desempacotada com sucesso")
print(f"- N_CLASSES    : {N_CLASSES}")
print(f"- INPUT_SHAPE  : {INPUT_SHAPE}")
print(f"- #clients     : {len(client_ids)}")
print(f"- train global : {len(train_dataset_global)}")
print(f"- test global  : {len(test_dataset_global)}")



[UCI-HAR] split=train | signal_set=all_9ch | X.shape=(7352, 128, 9)
[UCI-HAR] split=test | signal_set=all_9ch | X.shape=(2947, 128, 9)
✅ UCI-HAR carregado com sucesso
 - signal_set     : all_9ch
 - tensor_format  : channels_first
 - X_train.shape  : (7352, 9, 128)
 - X_test.shape   : (2947, 9, 128)
 - input_shape    : (9, 128)
 - n_classes      : 6
✅ Dataset ativo carregado com sucesso
 - dataset_name           : UCI_HAR
 - input_shape            : (9, 128)
 - n_classes              : 6
 - signal_set             : all_9ch
 - n_clients              : 21
 - train_global_samples   : 7352
 - test_global_samples    : 2947
✅ Interface do dataset desempacotada com sucesso
- N_CLASSES    : 6
- INPUT_SHAPE  : (9, 128)
- #clients     : 21
- train global : 7352
- test global  : 2947


In [28]:
# ID = 13 — Criação dos CLIENT_LOADERS
# ------------------------------------

BATCH_SIZE_TRAIN = FL_CFG["batch_size_train"]
BATCH_SIZE_EVAL = FL_CFG["batch_size_eval"]
NUM_WORKERS = EXPERIMENT_CFG.get("num_workers", 0)

CLIENT_TRAIN_LOADERS = [
    DataLoader(
        ds,
        batch_size=BATCH_SIZE_TRAIN,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )
    for ds in client_subsets_train
]

CLIENT_TEST_LOADERS = [
    DataLoader(
        ds,
        batch_size=BATCH_SIZE_EVAL,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )
    for ds in client_subsets_test
]

GLOBAL_TEST_LOADER = DataLoader(
    optional_global_eval_loader_dataset,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

print("✅ Loaders criados com sucesso")
print(f"- #CLIENT_TRAIN_LOADERS : {len(CLIENT_TRAIN_LOADERS)}")
print(f"- #CLIENT_TEST_LOADERS  : {len(CLIENT_TEST_LOADERS)}")
print(f"- BATCH_SIZE_TRAIN      : {BATCH_SIZE_TRAIN}")
print(f"- BATCH_SIZE_EVAL       : {BATCH_SIZE_EVAL}")
print(f"- NUM_WORKERS           : {NUM_WORKERS}")

if len(CLIENT_TRAIN_LOADERS) > 0:
    xb, yb = next(iter(CLIENT_TRAIN_LOADERS[0]))
    print(f"- Exemplo batch treino  : X={tuple(xb.shape)}, y={tuple(yb.shape)}")

if len(CLIENT_TEST_LOADERS) > 0:
    xb, yb = next(iter(CLIENT_TEST_LOADERS[0]))
    print(f"- Exemplo batch teste   : X={tuple(xb.shape)}, y={tuple(yb.shape)}")

xb, yb = next(iter(GLOBAL_TEST_LOADER))
print(f"- Exemplo batch global  : X={tuple(xb.shape)}, y={tuple(yb.shape)}")

✅ Loaders criados com sucesso
- #CLIENT_TRAIN_LOADERS : 21
- #CLIENT_TEST_LOADERS  : 21
- BATCH_SIZE_TRAIN      : 32
- BATCH_SIZE_EVAL       : 64
- NUM_WORKERS           : 0
- Exemplo batch treino  : X=(32, 9, 128), y=(32,)
- Exemplo batch teste   : X=(64, 9, 128), y=(64,)
- Exemplo batch global  : X=(64, 9, 128), y=(64,)


In [29]:
# ID = 14 — Diagnóstico do split por cliente
# ------------------------------------------

def _subset_labels(subset):
    if isinstance(subset, Subset):
        base_ds = subset.dataset
        idxs = np.array(subset.indices, dtype=int)
        return np.asarray(base_ds.y)[idxs]
    else:
        return np.asarray(subset.y)

rows = []

for cid, tr_subset, te_subset in zip(client_ids, client_subsets_train, client_subsets_test):
    y_tr = _subset_labels(tr_subset)
    y_te = _subset_labels(te_subset)

    rows.append({
        "client_id": int(cid),
        "n_train": int(len(tr_subset)),
        "n_test": int(len(te_subset)),
        "classes_train": int(len(np.unique(y_tr))) if len(y_tr) > 0 else 0,
        "classes_test": int(len(np.unique(y_te))) if len(y_te) > 0 else 0,
    })

client_split_df = pd.DataFrame(rows).sort_values("client_id").reset_index(drop=True)

client_n_train_samples = client_split_df["n_train"].tolist()
client_n_test_samples = client_split_df["n_test"].tolist()

print("✅ Diagnóstico do split por cliente")
display(client_split_df)

print("\nResumo:")
print(f"- #clients          : {len(client_ids)}")
print(f"- train total       : {sum(client_n_train_samples)}")
print(f"- test total        : {sum(client_n_test_samples)}")
print(f"- menor n_train     : {client_split_df['n_train'].min()}")
print(f"- maior n_train     : {client_split_df['n_train'].max()}")
print(f"- menor n_test      : {client_split_df['n_test'].min()}")
print(f"- maior n_test      : {client_split_df['n_test'].max()}")

✅ Diagnóstico do split por cliente


,client_id,n_train,n_test,classes_train,classes_test
0,1,277,70,6,6
1,3,272,69,6,6
2,5,241,61,6,6
3,6,260,65,6,6
4,7,246,62,6,6
5,8,224,57,6,6
6,11,252,64,6,6
7,14,258,65,6,6
8,15,262,66,6,6
9,16,292,74,6,6



Resumo:
- #clients          : 21
- train total       : 5871
- test total        : 1481
- menor n_train     : 224
- maior n_train     : 327
- menor n_test      : 57
- maior n_test      : 82


In [30]:
# ID = 15 — Função de treino local FedAvg (genérica, com histórico por época)
# ---------------------------------------------------------------------

def train_local_model(
    model: nn.Module,
    train_loader: DataLoader,
    n_epochs: int,
    lr: float,
    weight_decay: float = 0.0,
    max_grad_norm: float | None = None,
    device: str = DEVICE,
    desc: str = "local-train",
    eval_loader: DataLoader | None = None,
    track_metrics: bool = False,
) -> Dict[str, Any]:
    """
    Treina um modelo localmente em train_loader por n_epochs.

    Se track_metrics=True, avalia a cada época em:
      - train_loader (privado)
      - eval_loader (tipicamente test_loader_global), se fornecido

    Retorna:
      {
        "last_train_loss": float,
        "n_epochs": int,
        "history": [
            {
              "epoch": int,
              "train_loss": float,
              "acc_train": float,
              "f1_train_macro": float,
              "acc_eval": float | None,
              "f1_eval_macro": float | None,
            },
            ...
        ]
      }
    """
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    criterion = nn.CrossEntropyLoss()

    history: List[Dict[str, Any]] = []

    for epoch in range(1, n_epochs + 1):
        running_loss = 0.0
        n_batches = 0

        #for xb, yb in tqdm(train_loader, desc=f"{desc} [epoch {epoch}/{n_epochs}]", leave=False):
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=False)
            yb = yb.to(device, non_blocking=False)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()

            if max_grad_norm is not None and max_grad_norm > 0.0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

        avg_loss = running_loss / max(1, n_batches)
        #print(f"[{desc}] epoch {epoch}/{n_epochs} - loss={avg_loss:.4f}")

        record_epoch: Dict[str, Any] = {
            "epoch": epoch,
            "train_loss": float(avg_loss),
            "acc_train": None,
            "f1_train_macro": None,
            "acc_eval": None,
            "f1_eval_macro": None,
        }

        if track_metrics:
            # métricas em train (privado)
            m_train = evaluate_model(model, train_loader, device=device)
            record_epoch["acc_train"] = float(m_train["acc"])
            record_epoch["f1_train_macro"] = float(m_train["f1_macro"])

            # métricas em eval_loader (tipicamente test_loader_global)
            if eval_loader is not None:
                m_eval = evaluate_model(model, eval_loader, device=device)
                record_epoch["acc_eval"] = float(m_eval["acc"])
                record_epoch["f1_eval_macro"] = float(m_eval["f1_macro"])

        history.append(record_epoch)

    return {
        "last_train_loss": float(avg_loss),
        "n_epochs": n_epochs,
        "history": history,
    }


In [31]:
# ID = 16 — Função de treino local FedAvg + Fuzzy
# ------------------------------------------------

def train_local_model_AvgFuzzy(
    model: nn.Module,
    train_loader: DataLoader,
    n_epochs: int,
    lr: float,
    weight_decay: float = 0.0,
    max_grad_norm: float | None = None,
    device: str = DEVICE,
    desc: str = "local-train",
    eval_loader: DataLoader | None = None,
    track_metrics: bool = True,
) -> Dict[str, Any]:
    """
    Treina um modelo localmente em train_loader por n_epochs.

    Se track_metrics=True, avalia a cada época em:
      - train_loader (privado)
      - eval_loader (tipicamente test_loader_global), se fornecido

    Retorna:
      {
        "last_train_loss": float,
        "n_epochs": int,
        "history": [
            {
              "epoch": int,
              "train_loss": float,
              "acc_train": float,
              "f1_train_macro": float,
              "acc_eval": float | None,
              "f1_eval_macro": float | None,
            },
            ...
        ]
      }
    """
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    criterion = nn.CrossEntropyLoss()

    history: List[Dict[str, Any]] = []

    for epoch in range(1, n_epochs + 1):
        print(f"função de treinamento local: epoca {epoch}, lr {lr}")
        running_loss = 0.0
        n_batches = 0

        #for xb, yb in tqdm(train_loader, desc=f"{desc} [epoch {epoch}/{n_epochs}]", leave=False):
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=False)
            yb = yb.to(device, non_blocking=False)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()

            if max_grad_norm is not None and max_grad_norm > 0.0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

        avg_loss = running_loss / max(1, n_batches)
        #print(f"[{desc}] epoch {epoch}/{n_epochs} - loss={avg_loss:.4f}")

        record_epoch: Dict[str, Any] = {
            "epoch": epoch,
            "train_loss": float(avg_loss),
            "acc_train": None,
            "f1_train_macro": None,
            "acc_eval": None,
            "f1_eval_macro": None,
            "lr": float(lr),
        }

        if track_metrics:
            # métricas em train (privado)
            m_train = evaluate_model(model, train_loader, device=device)
            record_epoch["acc_train"] = float(m_train["acc"])
            record_epoch["f1_train_macro"] = float(m_train["f1_macro"])

            # métricas em eval_loader (tipicamente test_loader_global)
            if eval_loader is not None:
                m_eval = evaluate_model(model, eval_loader, device=device)
                record_epoch["acc_eval"] = float(m_eval["acc"])
                record_epoch["f1_eval_macro"] = float(m_eval["f1_macro"])

            record_epoch["lr"] = float(lr)
            history.append(record_epoch)

            lr = learning_rate_fuzzy(record_epoch["f1_train_macro"],record_epoch["train_loss"])


    return {
        "last_train_loss": float(avg_loss),
        "n_epochs": n_epochs,
        "history": history,
        "final_lr": float(lr),
    }


In [32]:
# ID = 17 — Hiperparâmetros federados (FL-HAR)
# --------------------------------------------

N_ROUNDS = FL_CFG["n_rounds"]
LOCAL_EPOCHS = FL_CFG["local_epochs"]
LR = FL_CFG["lr"]
WEIGHT_DECAY = FL_CFG["weight_decay"]
MAX_GRAD_NORM = FL_CFG["max_grad_norm"]
FEDPROX_MU = FL_CFG["fedprox_mu"]

print("✅ Hiperparâmetros federados carregados")
print(f"- N_ROUNDS      = {N_ROUNDS}")
print(f"- LOCAL_EPOCHS  = {LOCAL_EPOCHS}")
print(f"- LR            = {LR}")
print(f"- WEIGHT_DECAY  = {WEIGHT_DECAY}")
print(f"- MAX_GRAD_NORM = {MAX_GRAD_NORM}")
print(f"- FEDPROX_MU    = {FEDPROX_MU}")

✅ Hiperparâmetros federados carregados
- N_ROUNDS      = 10
- LOCAL_EPOCHS  = 5
- LR            = 0.0001
- WEIGHT_DECAY  = 0.01
- MAX_GRAD_NORM = 5.0
- FEDPROX_MU    = 0.01


In [33]:
# ID = 18 — CNN-1D Canônica (Baseline FedAvg / FedProx)
# -----------------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F


class CNN1DCanonical(nn.Module):
    """
    CNN-1D canônica para HAR, usada como baseline homogêneo
    em FedAvg e FedProx.

    Observação metodológica:
    - Esta arquitetura é única e compartilhada por todos os clientes.
    - Portanto, ela representa um baseline federado homogêneo.

    """

    def __init__(
        self,
        n_channels: int,
        n_classes: int,
        conv1_out: int = 64,
        conv2_out: int = 128,
        hidden_dim: int = 128,
        dropout: float = 0.5,
    ):
        super().__init__()

        self.n_channels = n_channels
        self.n_classes = n_classes
        self.conv1_out = conv1_out
        self.conv2_out = conv2_out
        self.hidden_dim = hidden_dim
        self.dropout_p = dropout

        # Bloco Conv 1
        self.conv1 = nn.Conv1d(
            in_channels=n_channels,
            out_channels=conv1_out,
            kernel_size=5,
            padding=2
        )
        self.bn1 = nn.BatchNorm1d(conv1_out)
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        # Bloco Conv 2
        self.conv2 = nn.Conv1d(
            in_channels=conv1_out,
            out_channels=conv2_out,
            kernel_size=3,
            padding=1
        )
        self.bn2 = nn.BatchNorm1d(conv2_out)

        # Classificador
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(conv2_out, hidden_dim)
        self.dropout = nn.Dropout(p=dropout)
        self.fc_out = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        # Esperado: x com shape [B, C, T]
        if x.ndim != 3:
            raise ValueError(f"Esperado tensor 3D [B, C, T], mas recebido shape={tuple(x.shape)}")

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))

        x = self.gap(x).squeeze(-1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc_out(x)
        return x

    def get_config(self):
        return {
            "model_name": "CNN1DCanonical",
            "n_channels": self.n_channels,
            "n_classes": self.n_classes,
            "conv1_out": self.conv1_out,
            "conv2_out": self.conv2_out,
            "hidden_dim": self.hidden_dim,
            "dropout": self.dropout_p,
            "architecture_type": "homogeneous_baseline",
        }


# ----------------------------------------------------
# Helper opcional para instanciar baseline canônico
# ----------------------------------------------------
def build_canonical_federated_model(input_shape, n_classes):
    """
    Cria a CNN-1D canônica para FedAvg/FedProx.

    input_shape esperado: (C, T)
    """
    if not isinstance(input_shape, (tuple, list)) or len(input_shape) != 2:
        raise ValueError(f"input_shape inválido para CNN1DCanonical: {input_shape}")

    n_channels = int(input_shape[0])

    model = CNN1DCanonical(
        n_channels=n_channels,
        n_classes=int(n_classes),
    )

    return model


In [34]:
#ID = 19
# -------------------------------------------------

def build_cnn1d_canonical(input_shape, n_classes):
    """
    Wrapper compatível com o notebook atual.
    Cria a CNN-1D canônica usada nos baselines FedAvg / FedProx.

    input_shape esperado: (C, T)
    """
    if not isinstance(input_shape, (tuple, list)) or len(input_shape) != 2:
        raise ValueError(f"input_shape inválido: {input_shape}")

    n_channels, _ = input_shape  # (C, T)

    return CNN1DCanonical(
        n_channels=int(n_channels),
        n_classes=int(n_classes)
    )

In [35]:
#ID = 20
# -------------------------------------------------
model = build_cnn1d_canonical(
    input_shape=INPUT_SHAPE,
    n_classes=N_CLASSES
)

n_params = sum(p.numel() for p in model.parameters())

BASELINE_MODEL_INFO = {
    "model_name": "CNN1DCanonical",
    "input_shape": tuple(INPUT_SHAPE),
    "n_classes": int(N_CLASSES),
    "n_params": int(n_params),
    "architecture_type": "homogeneous_baseline",
}

print(f"✅ Modelo baseline instanciado: {BASELINE_MODEL_INFO['model_name']}")
print(f" - input_shape        = {BASELINE_MODEL_INFO['input_shape']}")
print(f" - n_classes          = {BASELINE_MODEL_INFO['n_classes']}")
print(f" - n_params           = {BASELINE_MODEL_INFO['n_params']:,}")
print(f" - architecture_type  = {BASELINE_MODEL_INFO['architecture_type']}")

✅ Modelo baseline instanciado: CNN1DCanonical
 - input_shape        = (9, 128)
 - n_classes          = 6
 - n_params           = 45,318
 - architecture_type  = homogeneous_baseline


In [36]:
#ID = 21
# -------------------------------------------------
def fedavg_aggregate(local_weights, local_sizes):
    """
    Agregação FedAvg com suporte a parâmetros float e não-float.

    Parâmetros
    ----------
    local_weights : list[dict[str, torch.Tensor]]
        Lista de state_dicts locais.
    local_sizes : list[int]
        Número de amostras de treino usadas por cada cliente.

    Observação metodológica
    -----------------------
    local_sizes deve refletir apenas o conjunto de treino de cada cliente.
    Não usar contagens misturando treino e teste.
    """
    assert len(local_weights) == len(local_sizes), \
        "local_weights e local_sizes devem ter o mesmo comprimento."
    assert len(local_weights) > 0, "local_weights não pode ser vazio."
    assert all(size >= 0 for size in local_sizes), "local_sizes contém valor negativo."

    total_samples = sum(local_sizes)
    assert total_samples > 0, "A soma de local_sizes deve ser > 0."

    agg_weights = {}

    for k in local_weights[0].keys():
        ref_tensor = local_weights[0][k]

        # Parâmetros não float: copiar diretamente
        if not torch.is_floating_point(ref_tensor):
            agg_weights[k] = ref_tensor.clone()
            continue

        # Inicialização
        agg_weights[k] = torch.zeros_like(ref_tensor)

        # Média ponderada
        for w, size in zip(local_weights, local_sizes):
            agg_weights[k] += (size / total_samples) * w[k]

    return agg_weights

In [37]:
# ID = 22 — Treino local FedProx
# ------------------------------

def train_local_model_fedprox(
    model,
    global_model,
    train_loader,
    test_loader=None,
    epochs=1,
    lr=1e-3,
    mu=0.01,
    weight_decay=1e-4,
    device="cpu",
    max_grad_norm=None,
    verbose=False,
):
    """
    Treino local com penalização proximal do FedProx.

    A função retorna:
    - state_dict do modelo local treinado
    - métricas finais de treino
    - métricas finais de teste (se test_loader for fornecido)
    """
    model = model.to(device)
    global_model = global_model.to(device)

    model.train()
    global_params = {
        name: param.detach().clone().to(device)
        for name, param in global_model.named_parameters()
    }

    criterion = nn.CrossEntropyLoss()

    # Padronizado com o treino local genérico
    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    train_history = []

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)
            loss_ce = criterion(logits, yb)

            prox_term = 0.0
            for name, param in model.named_parameters():
                prox_term = prox_term + torch.sum((param - global_params[name]) ** 2)

            loss = loss_ce + (mu / 2.0) * prox_term

            loss.backward()

            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            running_correct += (preds == yb).sum().item()
            running_total += yb.size(0)

        epoch_loss = running_loss / max(running_total, 1)
        epoch_acc = running_correct / max(running_total, 1)

        train_history.append({
            "epoch": epoch + 1,
            "loss": epoch_loss,
            "acc": epoch_acc,
        })

        if verbose:
            print(
                f"[FedProx][Epoch {epoch+1}/{epochs}] "
                f"loss={epoch_loss:.4f} | acc={epoch_acc:.4f}"
            )

    final_train_metrics = evaluate_model(model, train_loader, device=device)

    final_test_metrics = None
    if test_loader is not None:
        final_test_metrics = evaluate_model(model, test_loader, device=device)

    return {
        "state_dict": copy.deepcopy(model.state_dict()),
        "train_metrics": final_train_metrics,
        "test_metrics": final_test_metrics,
        "train_history": train_history,
    }

In [38]:
# ID = 23 — FedAvg (CNN-1D canônica)
# ---------------------------------------------
# criando o modelo global
global_model = build_cnn1d_canonical(
    input_shape=INPUT_SHAPE,
    n_classes=N_CLASSES
).to(DEVICE)

global_weights = copy.deepcopy(global_model.state_dict())

fedavg_summary = []
fedavg_device_records = []

for r in range(1, N_ROUNDS + 1):
    print(f"\n========== FedAvg — ROUND {r} ==========")

    local_weights = []
    local_sizes = []

    # -------------------------------------------------
    # 1) Treino local em dados de TREINO
    # -------------------------------------------------
    for idx, did in enumerate(client_ids):
        train_loader = CLIENT_TRAIN_LOADERS[idx]

        assert train_loader is not None, f"FedAvg round {r} | train_loader ausente para client {did}"
        assert len(train_loader.dataset) > 0, f"FedAvg round {r} | sem treino para client {did}"

        #cria o modelo local
        local_model = build_cnn1d_canonical(
            input_shape=INPUT_SHAPE,
            n_classes=N_CLASSES
        ).to(DEVICE)

        local_model.load_state_dict(global_weights)

        train_stats = train_local_model(
            model=local_model,
            train_loader=train_loader,
            n_epochs=LOCAL_EPOCHS,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            max_grad_norm=MAX_GRAD_NORM,
            device=DEVICE,
            desc=f"fedavg_r{r}_client{did:02d}",
            eval_loader=None,
            track_metrics=False
        )

        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(int(len(train_loader.dataset)))


    # -------------------------------------------------
    # 2) Agregação FedAvg
    # -------------------------------------------------
    global_weights = fedavg_aggregate(local_weights, local_sizes)
    global_model.load_state_dict(global_weights)


    # -------------------------------------------------
    # 3) Avaliação do modelo global em TESTE por cliente /acontece a cada rodada
    # -------------------------------------------------
    acc_list = []
    f1_list = []
    precision_list = []
    recall_list = []


    for idx, did in enumerate(client_ids):
        train_loader = CLIENT_TRAIN_LOADERS[idx]
        test_loader = CLIENT_TEST_LOADERS[idx]

        n_train = int(len(train_loader.dataset))
        n_test = int(len(test_loader.dataset))

        assert test_loader is not None, f"FedAvg round {r} | test_loader ausente para client {did}"
        assert len(test_loader.dataset) > 0, f"FedAvg round {r} | sem teste para client {did}"


        metrics_train = evaluate_model(global_model,train_loader,device=DEVICE)
        metrics_test = evaluate_model(global_model,test_loader,device=DEVICE)

        acc_train = float(metrics_train["acc"])
        f1_train = float(metrics_train["f1_macro"])

        acc_test = float(metrics_test["acc"])
        f1_test = float(metrics_test["f1_macro"])

        gap_acc = acc_train - acc_test

        precision_train = float(metrics_train["precision_macro"])
        recall_train = float(metrics_train["recall_macro"])

        precision_test = float(metrics_test["precision_macro"])
        recall_test = float(metrics_test["recall_macro"])

        acc_list.append(acc_test)
        f1_list.append(f1_test)
        precision_list.append(precision_test)
        recall_list.append(recall_test)

        fedavg_device_records.append({
            "dataset": EXPERIMENT_CFG["dataset"],
            "method": "FedAvg",
            "round": int(r),
            "device_id": did,

            "n_train_samples": n_train,
            "n_test_samples": n_test,
            "n_eval_samples": n_test,

            "train_loader_type": "CLIENT_TRAIN_LOADERS",
            "test_loader_type": "CLIENT_TEST_LOADERS",
            "eval_split": "test",

            "local_epochs": int(LOCAL_EPOCHS),
            "lr": float(LR),
            "weight_decay": float(WEIGHT_DECAY),

            "acc_train": acc_train,
            "f1_macro_train": f1_train,
            "acc_test": acc_test,
            "f1_macro_test": f1_test,
            "gap_acc_train_test": float(gap_acc),
            "precision_train": precision_train,
            "recall_train": recall_train,
            "precision_test": precision_test,
            "recall_test": recall_test,
        })

    # -------------------------------------------------
    # 4) Agregados de fairness e desempenho
    # -------------------------------------------------
    acc_arr = np.array(acc_list, dtype=float)
    f1_arr = np.array(f1_list, dtype=float)
    precision_arr = np.array(precision_list, dtype=float)
    recall_arr = np.array(recall_list, dtype=float)

    acc_mean = float(np.mean(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_std = float(np.std(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_min = float(np.min(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_max = float(np.max(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_range = float(acc_max - acc_min) if acc_arr.size > 0 else float("nan")
    jain = jain_index(acc_arr)

    precision_mean = float(np.mean(precision_arr)) if precision_arr.size > 0 else float("nan")
    recall_mean = float(np.mean(recall_arr)) if recall_arr.size > 0 else float("nan")
    f1_mean = float(np.mean(f1_arr)) if f1_arr.size > 0 else float("nan")

    worst_idx = int(np.argmin(acc_arr)) if acc_arr.size > 0 else None
    best_idx = int(np.argmax(acc_arr)) if acc_arr.size > 0 else None

    worst_device_id = int(client_ids[worst_idx]) if worst_idx is not None else None
    best_device_id = int(client_ids[best_idx]) if best_idx is not None else None

    fedavg_summary.append({
        "dataset": EXPERIMENT_CFG["dataset"],
        "method": "FedAvg",
        "round": int(r),
        "eval_split": "test",

        "n_devices": int(len(client_ids)),
        "local_epochs": int(LOCAL_EPOCHS),

        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "acc_min": acc_min,
        "acc_max": acc_max,
        "acc_range": acc_range,
        "jain": jain,
        "precision_mean": precision_mean,
        "recall_mean": recall_mean,
        "f1_mean": f1_mean,

        "worst_device_id": worst_device_id,
        "best_device_id": best_device_id,
    })

    print(
        f"[FedAvg] TEST acc_mean={acc_mean:.4f}, "
        f"acc_min={acc_min:.4f}, acc_max={acc_max:.4f}, "
        f"range={acc_range:.4f}, Jain={jain:.4f} "
        f"precision_mean={precision_mean:.4f}, recall_mean={recall_mean:.4f}, f1_mean={f1_mean:.4f}"
    )


# -------------------------------------------------
# DataFrames + checks
# -------------------------------------------------
fedavg_summary_df = pd.DataFrame(fedavg_summary)
fedavg_device_df = pd.DataFrame(fedavg_device_records)

assert not fedavg_summary_df.empty, "fedavg_summary_df ficou vazio."
assert not fedavg_device_df.empty, "fedavg_device_df ficou vazio."
assert (fedavg_device_df["eval_split"] == "test").all(), "FedAvg não está sendo avaliado integralmente em teste."
assert (fedavg_device_df["n_test_samples"] > 0).all(), "Há client sem teste no FedAvg."
assert np.allclose(
    fedavg_device_df["gap_acc_train_test"].values,
    fedavg_device_df["acc_train"].values - fedavg_device_df["acc_test"].values
), "gap_acc_train_test inconsistente no FedAvg."

# -------------------------------------------------
# Salvamento
# -------------------------------------------------
fedavg_summary_csv = SPREADSHEETS_DIR / "fedavg_summary.csv"
fedavg_device_csv = SPREADSHEETS_DIR / "fedavg_device_metrics.csv"

fedavg_summary_df.to_csv(fedavg_summary_csv, index=False)
fedavg_device_df.to_csv(fedavg_device_csv, index=False)

print(f"✅ FedAvg summary salvo em: {fedavg_summary_csv}")
print(f"✅ FedAvg por client salvo em: {fedavg_device_csv}")


========== FedAvg — ROUND 1 ==========
[FedAvg] TEST acc_mean=0.6993, acc_min=0.5541, acc_max=0.8734, range=0.3194, Jain=0.9836 precision_mean=0.7183, recall_mean=0.6940, f1_mean=0.6437

========== FedAvg — ROUND 2 ==========
[FedAvg] TEST acc_mean=0.7948, acc_min=0.6154, acc_max=0.9848, range=0.3695, Jain=0.9762 precision_mean=0.8224, recall_mean=0.7932, f1_mean=0.7644

========== FedAvg — ROUND 3 ==========
[FedAvg] TEST acc_mean=0.8214, acc_min=0.5946, acc_max=0.9848, range=0.3903, Jain=0.9810 precision_mean=0.8485, recall_mean=0.8223, f1_mean=0.8014

========== FedAvg — ROUND 4 ==========
[FedAvg] TEST acc_mean=0.8282, acc_min=0.5946, acc_max=0.9848, range=0.3903, Jain=0.9814 precision_mean=0.8612, recall_mean=0.8295, f1_mean=0.8080

========== FedAvg — ROUND 5 ==========
[FedAvg] TEST acc_mean=0.8364, acc_min=0.5946, acc_max=1.0000, range=0.4054, Jain=0.9818 precision_mean=0.8725, recall_mean=0.8366, f1_mean=0.8185

========== FedAvg — ROUND 6 ==========
[FedAvg] TEST acc_mean=0.

In [39]:
# ID = 24 — FedAvg + Fuzzy (CNN-1D canônica)
# ---------------------------------------------
# criando o modelo global
global_model = build_cnn1d_canonical(
    input_shape=INPUT_SHAPE,
    n_classes=N_CLASSES
).to(DEVICE)

global_weights = copy.deepcopy(global_model.state_dict())

fedavg_fuzzy_summary = []
fedavg_fuzzy_device_records = []

lrs = [LR]*len(client_ids)
epochs = [LOCAL_EPOCHS]*len(client_ids)
losses = [0]*len(client_ids)
fedavg_fuzzy_lr_records = []
fedavg_fuzzy_epochs_records = []

for r in range(1, N_ROUNDS + 1):
    print(f"\n========== FedAvg+Fuzzy — ROUND {r} ==========")

    local_weights = []
    local_sizes = []

    print("local_sizes",local_sizes)

    # -------------------------------------------------
    # 1) Treino local em dados de TREINO
    # -------------------------------------------------
    for idx, did in enumerate(client_ids):
        train_loader = CLIENT_TRAIN_LOADERS[idx]

        assert train_loader is not None, f"FedAvg+Fuzzy round {r} | train_loader ausente para client {did}"
        assert len(train_loader.dataset) > 0, f"FedAvg+Fuzzy round {r} | sem treino para client {did}"

        local_model = build_cnn1d_canonical(
            input_shape=INPUT_SHAPE,
            n_classes=N_CLASSES
        ).to(DEVICE)

        local_model.load_state_dict(global_weights)

        train_stats = train_local_model_AvgFuzzy(
            model=local_model,
            train_loader=train_loader,
            n_epochs=epochs[idx],
            lr=lrs[idx],
            weight_decay=WEIGHT_DECAY,
            max_grad_norm=MAX_GRAD_NORM,
            device=DEVICE,
            desc=f"fedavg_r{r}_client{did:02d}",
            eval_loader=None,
            track_metrics=True
        )

        for h in train_stats["history"]:
          fedavg_fuzzy_lr_records.append({
              "round": r,
              "device_id": did,
              "epoch": h["epoch"],
              "lr": h["lr"],
              "loss": h["train_loss"],
              "acc_train": h["acc_train"]

        })


        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(int(len(train_loader.dataset)))

        lrs[idx] = train_stats["final_lr"]

        losses[idx]=train_stats['last_train_loss']
        print( f"fim da rodada local para o cliente {did}")
        print()


    # -------------------------------------------------
    # 2) Agregação FedAvg
    # -------------------------------------------------
    global_weights = fedavg_aggregate(local_weights, local_sizes)
    global_model.load_state_dict(global_weights)


    # -------------------------------------------------
    # 3) Avaliação do modelo global em TESTE por cliente
    # -------------------------------------------------
    acc_list = []
    f1_list = []
    precision_list = []
    recall_list = []

    for idx, did in enumerate(client_ids):
        train_loader = CLIENT_TRAIN_LOADERS[idx]
        test_loader = CLIENT_TEST_LOADERS[idx]

        n_train = int(len(train_loader.dataset))
        n_test = int(len(test_loader.dataset))

        assert test_loader is not None, f"FedAvg round {r} | test_loader ausente para client {did}"
        assert len(test_loader.dataset) > 0, f"FedAvg round {r} | sem treino para client {did}"

        metrics_train = evaluate_model(global_model,train_loader,device=DEVICE)
        metrics_test = evaluate_model(global_model,test_loader,device=DEVICE)

        acc_train = float(metrics_train["acc"])
        f1_train = float(metrics_train["f1_macro"])

        acc_test = float(metrics_test["acc"])
        f1_test = float(metrics_test["f1_macro"])

        gap_acc = acc_train - acc_test

        precision_train = float(metrics_train["precision_macro"])
        recall_train = float(metrics_train["recall_macro"])

        precision_test = float(metrics_test["precision_macro"])
        recall_test = float(metrics_test["recall_macro"])

        acc_list.append(acc_test)
        f1_list.append(f1_test)
        precision_list.append(precision_test)
        recall_list.append(recall_test)

        fedavg_fuzzy_epochs_records.append({
            "round": r,
            "device_id": did,
            "epochs": epochs[idx],
            "gap_acc": gap_acc,
            "loss": losses[idx]
})

        print(f"epoca do cliente {did} antes da atualização no round {r} = {epochs[idx]}")
        epochs[idx] = int(epochs_fuzzy(gap_acc,losses[idx]))

        print(f"epoca do cliente {did} depois da atualização no round {r} = {epochs[idx]}")
        print()


        fedavg_fuzzy_device_records.append({
            "dataset": EXPERIMENT_CFG["dataset"],
            "method": "FedAvgFuzzy",
            "round": int(r),
            "device_id": did,

            "n_train_samples": n_train,
            "n_test_samples": n_test,
            "n_eval_samples": n_test,

            "train_loader_type": "CLIENT_TRAIN_LOADERS",
            "test_loader_type": "CLIENT_TEST_LOADERS",
            "eval_split": "test",

            "local_epochs": int(epochs[idx]),
            "lr": float(lrs[idx]),
            "weight_decay": float(WEIGHT_DECAY),

            "acc_train": acc_train,
            "f1_macro_train": f1_train,
            "acc_test": acc_test,
            "f1_macro_test": f1_test,
            "gap_acc_train_test": float(gap_acc),
            "precision_train": precision_train,
            "recall_train": recall_train,
            "precision_test": precision_test,
            "recall_test": recall_test,
        })

    # -------------------------------------------------
    # 4) Agregados de fairness e desempenho
    # -------------------------------------------------
    acc_arr = np.array(acc_list, dtype=float)
    f1_arr = np.array(f1_list, dtype=float)
    precision_arr = np.array(precision_list, dtype=float)
    recall_arr = np.array(recall_list, dtype=float)

    acc_mean = float(np.mean(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_std = float(np.std(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_min = float(np.min(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_max = float(np.max(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_range = float(acc_max - acc_min) if acc_arr.size > 0 else float("nan")
    jain = jain_index(acc_arr)

    precision_mean = float(np.mean(precision_arr)) if precision_arr.size > 0 else float("nan")
    recall_mean = float(np.mean(recall_arr)) if recall_arr.size > 0 else float("nan")
    f1_mean = float(np.mean(f1_arr)) if f1_arr.size > 0 else float("nan")

    worst_idx = int(np.argmin(acc_arr)) if acc_arr.size > 0 else None
    best_idx = int(np.argmax(acc_arr)) if acc_arr.size > 0 else None

    worst_device_id = int(client_ids[worst_idx]) if worst_idx is not None else None
    best_device_id = int(client_ids[best_idx]) if best_idx is not None else None

    fedavg_fuzzy_summary.append({
        "dataset": EXPERIMENT_CFG["dataset"],
        "method": "FedAvgFuzzy",
        "round": int(r),
        "eval_split": "test",

        "n_devices": int(len(client_ids)),
        "local_epochs": int(epochs[idx]),

        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "acc_min": acc_min,
        "acc_max": acc_max,
        "acc_range": acc_range,
        "jain": jain,
        "precision_mean": precision_mean,
        "recall_mean": recall_mean,
        "f1_mean": f1_mean,

        "worst_device_id": worst_device_id,
        "best_device_id": best_device_id,
    })

    print(
        f"[FedAvg] TEST acc_mean={acc_mean:.4f}, "
        f"acc_min={acc_min:.4f}, acc_max={acc_max:.4f}, "
        f"range={acc_range:.4f}, Jain={jain:.4f}"
        f"precision_mean={precision_mean:.4f}, "
        f"recall_mean={recall_mean:.4f}, "
        f"f1_mean={f1_mean:.4f}"
    )



# -------------------------------------------------
# DataFrames + checks
# -------------------------------------------------
fedavg_fuzzy_lr_df = pd.DataFrame(fedavg_fuzzy_lr_records)
display(fedavg_fuzzy_lr_df.head())
fedavg_fuzzy_epochs_df = pd.DataFrame(fedavg_fuzzy_epochs_records)
display(fedavg_fuzzy_epochs_df.head())
fedavg_fuzzy_summary_df = pd.DataFrame(fedavg_fuzzy_summary)
fedavg_fuzzy_device_df = pd.DataFrame(fedavg_fuzzy_device_records)

assert not fedavg_fuzzy_summary_df.empty, "fedavg_summary_df ficou vazio."
assert not fedavg_fuzzy_device_df.empty, "fedavg_device_df ficou vazio."
assert (fedavg_fuzzy_device_df["eval_split"] == "test").all(), "FedAvg não está sendo avaliado integralmente em teste."
assert (fedavg_fuzzy_device_df["n_test_samples"] > 0).all(), "Há client sem teste no FedAvg."
assert np.allclose(
    fedavg_fuzzy_device_df["gap_acc_train_test"].values,
    fedavg_fuzzy_device_df["acc_train"].values - fedavg_fuzzy_device_df["acc_test"].values
), "gap_acc_train_test inconsistente no FedAvg."

# -------------------------------------------------
# Salvamento
# -------------------------------------------------
fedavg_fuzzy_summary_csv = SPREADSHEETS_DIR / "fedavgfuzzy_summary.csv"
fedavg_fuzzy_device_csv = SPREADSHEETS_DIR / "fedavgfuzzy_device_metrics.csv"

fedavg_fuzzy_summary_df.to_csv(fedavg_fuzzy_summary_csv, index=False)
fedavg_fuzzy_device_df.to_csv(fedavg_fuzzy_device_csv, index=False)

print(f"✅ FedAvg+Fuzzy summary salvo em: {fedavg_fuzzy_summary_csv}")
print(f"✅ FedAvg+Fuzzy por client salvo em: {fedavg_fuzzy_device_csv}")


========== FedAvg+Fuzzy — ROUND 1 ==========
local_sizes []
função de treinamento local: epoca 1, lr 0.0001
função de treinamento local: epoca 2, lr 0.008960262647734968
função de treinamento local: epoca 3, lr 0.008643283494352309
função de treinamento local: epoca 4, lr 0.008735850980749645
função de treinamento local: epoca 5, lr 0.006885663468248326
fim da rodada local para o cliente 1

função de treinamento local: epoca 1, lr 0.0001
função de treinamento local: epoca 2, lr 0.008785628008101322
função de treinamento local: epoca 3, lr 0.008635478158711818
função de treinamento local: epoca 4, lr 0.008807339409090488
função de treinamento local: epoca 5, lr 0.008975502522782427
fim da rodada local para o cliente 3

função de treinamento local: epoca 1, lr 0.0001
função de treinamento local: epoca 2, lr 0.00896381143544968
função de treinamento local: epoca 3, lr 0.008544507243061538
função de treinamento local: epoca 4, lr 0.008731110456458545
função de treinamento local: epoca 5, 

,round,device_id,epoch,lr,loss,acc_train
0,1,1,1,0.000100,1.772942,0.126354
1,1,1,2,0.008960,1.703183,0.288809
2,1,1,3,0.008643,1.636919,0.393502
3,1,1,4,0.008736,1.535482,0.736462
4,1,1,5,0.006886,1.487624,0.851986


,round,device_id,epochs,gap_acc,loss
0,1,1,5,0.004642,1.487624
1,1,3,5,-0.039695,1.440583
2,1,5,5,-0.022924,1.503306
3,1,6,5,-0.011538,1.493691
4,1,7,5,-0.007606,1.467337


✅ FedAvg+Fuzzy summary salvo em: /content/FL_Fuzzy_results/metricas/Exp_20260926_184858/arquivos_csv/fedavgfuzzy_summary.csv
✅ FedAvg+Fuzzy por client salvo em: /content/FL_Fuzzy_results/metricas/Exp_20260926_184858/arquivos_csv/fedavgfuzzy_device_metrics.csv


In [40]:
# ID = 25 — Gráfico LR contínuo por cliente
# ---------------------------------------------
fedavg_fuzzy_lr_df = pd.DataFrame(
    fedavg_fuzzy_lr_records
)

SAVE_DIR = GRAPHICS_DIR / "lr_por_cliente"
SAVE_DIR.mkdir(exist_ok=True)

plt.style.use("default")


for cliente in sorted(
    fedavg_fuzzy_lr_df["device_id"].unique()
):

    temp = (
        fedavg_fuzzy_lr_df[
            fedavg_fuzzy_lr_df["device_id"]
            == cliente
        ]
        .sort_values(
            [
                "round",
                "epoch"
            ]
        )
        .reset_index(drop=True)
    )

    temp["step"] = range(
        len(temp)
    )

    fig, ax = plt.subplots(
        figsize=(16,9),
        facecolor="white"
    )

    ax.set_facecolor("white")

    ax.plot(
        temp["step"],

        temp["lr"],

        linewidth=2,

        color="#1f77b4"
    )

    ax.set_title(
        f"Client {cliente}"
    )

    ax.set_xlabel(
        "Iteration"
    )

    ax.set_ylabel(
        "Learning Rate"
    )

    ax.grid(
        color="#BBBBBB",
        alpha=0.5
    )

    plt.tight_layout()

    nome = (
        SAVE_DIR
        /
        f"client_{cliente}_lr.png"
    )

    fig.savefig(
        nome,
        dpi=300,
        facecolor="white"
    )

    plt.close()


print("Gráficos salvos em:")
print(SAVE_DIR)

# -------------------------------------------------
# Gráfico Epocas locais por cliente
# -------------------------------------------------
SAVE_DIR = (
    GRAPHICS_DIR
    /
    "epocas_por_cliente"
)

SAVE_DIR.mkdir(
    exist_ok=True
)

plt.style.use("default")


for cliente in sorted(
    fedavg_fuzzy_epochs_df["device_id"].unique()
):

    temp = (
        fedavg_fuzzy_epochs_df[
            fedavg_fuzzy_epochs_df["device_id"]
            == cliente
        ]
        .sort_values(
            "round"
        )
    )

    fig, ax = plt.subplots(
        figsize=(16,9),
        facecolor="white"
    )

    ax.set_facecolor(
        "white"
    )

    ax.plot(

        temp["round"],

        temp["epochs"],

        marker="o",

        linewidth=2,

        color="#D62728"

    )

    ax.set_title(
        f"Client {cliente}"
    )

    ax.set_xlabel(
        "Round"
    )

    ax.set_ylabel(
        "Local Epochs"
    )

    ax.grid(
        color="#BBBBBB",
        alpha=0.5
    )

    plt.tight_layout()

    nome = (
        SAVE_DIR
        /
        f"client_{cliente}_epochs.png"
    )

    fig.savefig(
        nome,
        dpi=300,
        facecolor="white"
    )

    plt.close()


print("Gráficos salvos em:")
print(SAVE_DIR)


# -------------------------------------------------
# Salvar dados utilizados nos gráficos
# -------------------------------------------------

DATA_DIR = EXPERIMENT_DIR / "arquivos_csv"
DATA_DIR.mkdir(exist_ok=True)

# Dados do Learning Rate
fedavg_fuzzy_lr_df.to_csv(
    DATA_DIR / "fedavg_fuzzy_lr_por_cliente.csv",
    index=False
)

# Dados das épocas locais
fedavg_fuzzy_epochs_df.to_csv(
    DATA_DIR / "fedavg_fuzzy_epocas_por_cliente.csv",
    index=False
)

print("Dados dos gráficos salvos em:")
print(DATA_DIR)

Gráficos salvos em:
/content/FL_Fuzzy_results/metricas/Exp_20260926_184858/graficos/lr_por_cliente
Gráficos salvos em:
/content/FL_Fuzzy_results/metricas/Exp_20260926_184858/graficos/epocas_por_cliente
Dados dos gráficos salvos em:
/content/FL_Fuzzy_results/metricas/Exp_20260926_184858/arquivos_csv


In [41]:
# ID = 26 — FedProx (CNN-1D canônica)
# ----------------------------------------------
import torch.optim as optim
global_model = build_cnn1d_canonical(
    input_shape=INPUT_SHAPE,
    n_classes=N_CLASSES
).to(DEVICE)

global_weights = copy.deepcopy(global_model.state_dict())

fedprox_summary = []
fedprox_device_records = []

for r in range(1, N_ROUNDS + 1):
    print(f"\n========== FedProx — ROUND {r} ==========")

    local_weights = []
    local_sizes = []

    # -------------------------------------------------
    # 1) Treino local em dados de TREINO
    # -------------------------------------------------
    for idx, did in enumerate(client_ids):
        train_loader = CLIENT_TRAIN_LOADERS[idx]

        assert train_loader is not None, f"FedProx round {r} | train_loader ausente para client {did}"
        assert len(train_loader.dataset) > 0, f"FedProx round {r} | sem treino para client {did}"

        local_model = build_cnn1d_canonical(
            input_shape=INPUT_SHAPE,
            n_classes=N_CLASSES
        ).to(DEVICE)

        local_model.load_state_dict(global_weights)

        global_model_ref = build_cnn1d_canonical(
            input_shape=INPUT_SHAPE,
            n_classes=N_CLASSES
        ).to(DEVICE)
        global_model_ref.load_state_dict(global_weights)

        train_stats = train_local_model_fedprox(
            model=local_model,
            global_model=global_model_ref,
            train_loader=train_loader,
            test_loader=None,
            epochs=LOCAL_EPOCHS,
            lr=LR,
            mu=FEDPROX_MU,
            weight_decay=WEIGHT_DECAY,
            max_grad_norm=MAX_GRAD_NORM,
            device=DEVICE,
            verbose=False,
        )

        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(int(len(train_loader.dataset)))

    # -------------------------------------------------
    # 2) Agregação (mesma regra do FedAvg)
    # -------------------------------------------------
    global_weights = fedavg_aggregate(local_weights, local_sizes)
    global_model.load_state_dict(global_weights)

    # -------------------------------------------------
    # 3) Avaliação do modelo global em TESTE por cliente
    # -------------------------------------------------
    acc_list = []
    f1_list = []
    precision_list = []
    recall_list = []

    for idx, did in enumerate(client_ids):
        train_loader = CLIENT_TRAIN_LOADERS[idx]
        test_loader = CLIENT_TEST_LOADERS[idx]

        n_train = int(len(train_loader.dataset))
        n_test = int(len(test_loader.dataset))

        assert test_loader is not None, f"FedProx round {r} | test_loader ausente para client {did}"
        assert len(test_loader.dataset) > 0, f"FedProx round {r} | sem teste para client {did}"

        metrics_train = evaluate_model(global_model,train_loader,device=DEVICE)
        metrics_test = evaluate_model(global_model,test_loader,device=DEVICE)

        acc_train = float(metrics_train["acc"])
        f1_train = float(metrics_train["f1_macro"])

        acc_test = float(metrics_test["acc"])
        f1_test = float(metrics_test["f1_macro"])

        gap_acc = acc_train - acc_test

        precision_train = float(metrics_train["precision_macro"])
        recall_train = float(metrics_train["recall_macro"])

        precision_test = float(metrics_test["precision_macro"])
        recall_test = float(metrics_test["recall_macro"])

        acc_list.append(acc_test)
        f1_list.append(f1_test)
        precision_list.append(precision_test)
        recall_list.append(recall_test)

        fedprox_device_records.append({
            "dataset": EXPERIMENT_CFG["dataset"],
            "method": "FedProx",
            "round": int(r),
            "device_id": did,

            "n_train_samples": n_train,
            "n_test_samples": n_test,
            "n_eval_samples": n_test,

            "train_loader_type": "CLIENT_TRAIN_LOADERS",
            "test_loader_type": "CLIENT_TEST_LOADERS",
            "eval_split": "test",

            "local_epochs": int(LOCAL_EPOCHS),
            "lr": float(LR),
            "mu": float(FEDPROX_MU),
            "weight_decay": float(WEIGHT_DECAY),

            "acc_train": acc_train,
            "f1_macro_train": f1_train,
            "acc_test": acc_test,
            "f1_macro_test": f1_test,
            "gap_acc_train_test": float(gap_acc),
            "precision_train": precision_train,
            "recall_train": recall_train,
            "precision_test": precision_test,
            "recall_test": recall_test,
        })

    # -------------------------------------------------
    # 4) Agregados de fairness e desempenho
    # -------------------------------------------------
    acc_arr = np.array(acc_list, dtype=float)
    f1_arr = np.array(f1_list, dtype=float)
    precision_arr = np.array(precision_list, dtype=float)
    recall_arr = np.array(recall_list, dtype=float)

    acc_mean = float(np.mean(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_std = float(np.std(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_min = float(np.min(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_max = float(np.max(acc_arr)) if acc_arr.size > 0 else float("nan")
    acc_range = float(acc_max - acc_min) if acc_arr.size > 0 else float("nan")
    jain = jain_index(acc_arr)

    f1_mean = float(np.mean(f1_arr)) if f1_arr.size > 0 else float("nan")
    precision_mean = float(np.mean(precision_arr)) if precision_arr.size > 0 else float("nan")
    recall_mean = float(np.mean(recall_arr)) if recall_arr.size > 0 else float("nan")

    worst_idx = int(np.argmin(acc_arr)) if acc_arr.size > 0 else None
    best_idx = int(np.argmax(acc_arr)) if acc_arr.size > 0 else None

    worst_device_id = int(client_ids[worst_idx]) if worst_idx is not None else None
    best_device_id = int(client_ids[best_idx]) if best_idx is not None else None

    fedprox_summary.append({
        "dataset": EXPERIMENT_CFG["dataset"],
        "method": "FedProx",
        "round": int(r),
        "eval_split": "test",

        "n_devices": int(len(client_ids)),
        "local_epochs": int(LOCAL_EPOCHS),
        "mu": float(FEDPROX_MU),

        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "acc_min": acc_min,
        "acc_max": acc_max,
        "acc_range": acc_range,
        "jain": jain,
        "precision_mean": precision_mean,
        "recall_mean": recall_mean,
        "f1_mean": f1_mean,

        "worst_device_id": worst_device_id,
        "best_device_id": best_device_id,
    })

    print(
        f"[FedProx] TEST acc_mean={acc_mean:.4f}, "
        f"acc_min={acc_min:.4f}, acc_max={acc_max:.4f}, "
        f"range={acc_range:.4f}, Jain={jain:.4f}, "
        f"precision_mean={precision_mean:.4f}, "
        f"recall_mean={recall_mean:.4f}, "
        f"f1_mean={f1_mean:.4f} "
    )

# -------------------------------------------------
# DataFrames + checks
# -------------------------------------------------
fedprox_summary_df = pd.DataFrame(fedprox_summary)
fedprox_device_df = pd.DataFrame(fedprox_device_records)

assert not fedprox_summary_df.empty, "fedprox_summary_df ficou vazio."
assert not fedprox_device_df.empty, "fedprox_device_df ficou vazio."
assert (fedprox_device_df["eval_split"] == "test").all(), "FedProx não está sendo avaliado integralmente em teste."
assert (fedprox_device_df["n_test_samples"] > 0).all(), "Há client sem teste no FedProx."
assert np.allclose(
    fedprox_device_df["gap_acc_train_test"].values,
    fedprox_device_df["acc_train"].values - fedprox_device_df["acc_test"].values
), "gap_acc_train_test inconsistente no FedProx."

# -------------------------------------------------
# Salvamento
# -------------------------------------------------
fedprox_summary_csv = SPREADSHEETS_DIR  / "fedprox_summary.csv"
fedprox_device_csv = SPREADSHEETS_DIR / "fedprox_device_metrics.csv"

fedprox_summary_df.to_csv(fedprox_summary_csv, index=False)
fedprox_device_df.to_csv(fedprox_device_csv, index=False)

print(f"✅ FedProx summary salvo em: {fedprox_summary_csv}")
print(f"✅ FedProx por client salvo em: {fedprox_device_csv}")


========== FedProx — ROUND 1 ==========
[FedProx] TEST acc_mean=0.6664, acc_min=0.4865, acc_max=0.7927, range=0.3062, Jain=0.9824, precision_mean=0.6513, recall_mean=0.6599, f1_mean=0.6002 

========== FedProx — ROUND 2 ==========
[FedProx] TEST acc_mean=0.7724, acc_min=0.5902, acc_max=0.9420, range=0.3519, Jain=0.9857, precision_mean=0.8014, recall_mean=0.7757, f1_mean=0.7422 

========== FedProx — ROUND 3 ==========
[FedProx] TEST acc_mean=0.8110, acc_min=0.5738, acc_max=0.9688, range=0.3950, Jain=0.9803, precision_mean=0.8361, recall_mean=0.8108, f1_mean=0.7873 

========== FedProx — ROUND 4 ==========
[FedProx] TEST acc_mean=0.8117, acc_min=0.5246, acc_max=0.9844, range=0.4598, Jain=0.9756, precision_mean=0.8295, recall_mean=0.8101, f1_mean=0.7859 

========== FedProx — ROUND 5 ==========
[FedProx] TEST acc_mean=0.8171, acc_min=0.5000, acc_max=1.0000, range=0.5000, Jain=0.9732, precision_mean=0.8484, recall_mean=0.8153, f1_mean=0.7935 

========== FedProx — ROUND 6 ==========
[Fed

In [42]:
# ID = 27 — Comparação final FedAvg vs FedProx vs FedAvgFuzzy
# -----------------------------------------------------------

compare_cols = ["round", "acc_mean", "acc_std", "acc_min", "acc_max", "acc_range", "jain","f1_mean", "recall_mean","precision_mean"]

fedavg_cmp = fedavg_summary_df[compare_cols].copy()
fedavg_cmp["method"] = "FedAvg"

fedprox_cmp = fedprox_summary_df[compare_cols].copy()
fedprox_cmp["method"] = "FedProx"

fedfuzzy_cmp = fedavg_fuzzy_summary_df[compare_cols].copy()
fedfuzzy_cmp["method"] = "FedAvgFuzzy"

comparison_df = pd.concat([fedavg_cmp, fedprox_cmp,fedfuzzy_cmp], axis=0, ignore_index=True)
comparison_df = comparison_df[["method"] + compare_cols].sort_values(["round", "method"]).reset_index(drop=True)

display(comparison_df)

comparison_csv = SPREADSHEETS_DIR / "comparison.csv"

comparison_df.to_csv(comparison_csv, index=False)

print(f"✅ Comparação salva em: {comparison_csv}")

final_round_df = comparison_df[comparison_df["round"] == comparison_df["round"].max()].copy()
print("\nResumo da rodada final:")
display(final_round_df)

,method,round,acc_mean,acc_std,acc_min,acc_max,acc_range,jain,f1_mean,recall_mean,precision_mean
0,FedAvg,1,0.699278,0.090217,0.554054,0.873418,0.319364,0.983628,0.643713,0.693999,0.718308
1,FedAvgFuzzy,1,0.525768,0.057484,0.375000,0.600000,0.225000,0.988187,0.413109,0.531539,0.430547
2,FedProx,1,0.666435,0.089296,0.486486,0.792683,0.306196,0.982363,0.600170,0.659911,0.651324
3,FedAvg,2,0.794756,0.124130,0.615385,0.984848,0.369464,0.976187,0.764415,0.793207,0.822403
4,FedAvgFuzzy,2,0.703967,0.173160,0.229730,1.000000,0.770270,0.942947,0.662259,0.688742,0.740237
5,FedProx,2,0.772408,0.092903,0.590164,0.942029,0.351865,0.985740,0.742235,0.775707,0.801402
6,FedAvg,3,0.821401,0.114312,0.594595,0.984848,0.390254,0.981001,0.801364,0.822315,0.848550
7,FedAvgFuzzy,3,0.895853,0.125709,0.527027,1.000000,0.472973,0.980690,0.888805,0.897362,0.914250
8,FedProx,3,0.811019,0.114880,0.573770,0.968750,0.394980,0.980330,0.787347,0.810758,0.836085
9,FedAvg,4,0.828209,0.113962,0.594595,0.984848,0.390254,0.981418,0.808009,0.829519,0.861241


✅ Comparação salva em: /content/FL_Fuzzy_results/metricas/Exp_20260926_184858/arquivos_csv/comparison.csv

Resumo da rodada final:


,method,round,acc_mean,acc_std,acc_min,acc_max,acc_range,jain,f1_mean,recall_mean,precision_mean
27,FedAvg,10,0.873852,0.122066,0.540541,1.0,0.459459,0.980861,0.861684,0.873786,0.889241
28,FedAvgFuzzy,10,0.920470,0.106393,0.567568,1.0,0.432432,0.986816,0.918161,0.924421,0.940910
29,FedProx,10,0.867686,0.155095,0.445946,1.0,0.554054,0.969039,0.855352,0.869091,0.888963


In [43]:
# ID = 28 — Geração dos graficos de Comparação final FedAvg vs FedProx
# --------------------------------------------------------------------

plt.style.use("seaborn-v0_8")

style_map = {
    "FedAvg": {
        "color": "#00B2EE",
        "linestyle": "-",
        "marker": ""
    },


    "FedProx": {
        "color": "#912CEE",
        "linestyle": "--",
        "marker": ""
    },

    "FedAvg+Fuzzy": {
        "color": "#FF34B3",
        "linestyle": "-.",
        "marker": ""
    },
}

df_clients = pd.concat([fedavg_device_df, fedprox_device_df,fedavg_fuzzy_device_df],axis=0,ignore_index=True)
df_clients = df_clients.sort_values(["device_id", "round"])

print("Métodos: ", df_clients["method"].unique())
print("Clientes:", df_clients["device_id"].nunique())

SAVE_DIR = GRAPHICS_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Salvando em: {SAVE_DIR}")

clients = df_clients["device_id"].unique()

for client in clients:
    df_c = df_clients[df_clients["device_id"] == client]

    plt.figure(figsize=(14, 8))

    for method in df_clients["method"].unique():
        df_m = df_c[df_c["method"] == method]

        if df_m.empty:
            continue

        style = style_map.get(method, {})

        plt.plot(
            df_m["round"],
            df_m["acc_test"],
            label=method,
            linewidth=2,
            markersize=6,
            **style
        )

    plt.title(f"Cliente {client}", fontsize=14)
    plt.xlabel("Rodada", fontsize=12)
    plt.ylabel("Acurácia (teste)", fontsize=12)

    plt.legend()
    plt.grid(alpha=0.95)

    save_path = os.path.join(SAVE_DIR, f"client_{client}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.close()

print(f"Gráficos salvos com sucesso em: {SAVE_DIR}")

Métodos:  ['FedAvg' 'FedProx' 'FedAvgFuzzy']
Clientes: 21
Salvando em: /content/FL_Fuzzy_results/metricas/Exp_20260926_184858/graficos
Gráficos salvos com sucesso em: /content/FL_Fuzzy_results/metricas/Exp_20260926_184858/graficos
